# 2. World Bank Open Data
Endpoint:
https://api.worldbank.org/v2/country/all/indicator/SP.POP.TOTL?format=json&date=2000:2025&per_page=20000

In [0]:
%run ./config

In [0]:
WORLD_BANK_URL = ("https://api.worldbank.org/v2/"
                  "country/all/indicator/SP.POP.TOTL")

params = {"format": "json",
            "date": "2000:2025",
        "per_page": 20000}

response = http.get(WORLD_BANK_URL,params=params,timeout=120)
response.raise_for_status()
world_bank_data = response.json()
metadata = world_bank_data[0]
records = world_bank_data[1]

population_rows = []

In [0]:
for record in records:
    population_rows.append((INGESTION_ID,
                            record.get("countryiso3code"),
                            record.get("countryiso3code"),
                            int(record["date"]) if record.get("date") else None,
                            json.dumps(record, ensure_ascii=False),
                            response.url,
                            COLLECTED_AT))

print(f"Registros coletados: {len(population_rows)}")

In [0]:
population_schema = StructType([StructField("ingestion_id", StringType(), False),
                                StructField("country_id", StringType(), True),
                                StructField("country_iso3", StringType(), True),
                                StructField("year", IntegerType(), True),
                                StructField("payload", StringType(), False),
                                StructField("source_url", StringType(), False),
                                StructField("collected_at", TimestampType(), False)])

In [0]:
population_df = spark.createDataFrame(population_rows,schema=population_schema)
population_df.write.format("delta")\
                   .mode("overwrite")\
                   .option("overwriteSchema", "true")\
                   .saveAsTable(f"{CATALOG}.{SCHEMA}.brz_population")

In [0]:
%sql
describe extended mvp_eng_dados.mvp_cancer.brz_clinical_trials

In [0]:
%sql
SELECT * FROM mvp_eng_dados.mvp_cancer.brz_population